# HalluStream quickstart

A small probe reads Qwen3-4B's hidden states while it writes and flags the names, dates, places and titles it is probably making up. It generates no extra text and needs no web search. Results and method: [github.com/pranav-sethia/hallustream](https://github.com/pranav-sethia/hallustream).

**Runtime:** choose *Runtime → Change runtime type → T4 GPU*. The first run downloads Qwen3-4B (about 8 GB).

In [ ]:
!pip install -q "hallustream[entities] @ git+https://github.com/pranav-sethia/hallustream"

In [ ]:
import torch

from hallustream import Detector

# T4 GPUs lack fast bfloat16, so use float16 (the probe was evaluated in bfloat16 on an L4)
det = Detector.from_pretrained("qwen3-4b", dtype=torch.float16)

## Watch it write

Each sentence appears once it has been checked. Highlighted details are the ones the probe thinks the model is making up.

In [ ]:
from IPython.display import HTML, display

from hallustream import Report
from hallustream.detector import highlight


def watch(prompt):
    text, found, shown = "", [], display(HTML(""), display_id=True)
    for ev in det.stream(prompt, max_new_tokens=300):
        text += ev.text
        found += ev.entities
        if ev.entities or ev.done:
            body = highlight(text[: ev.checked], [e for e in found if e.flagged], "html")
            shown.update(HTML(f"<div style='white-space:pre-wrap;line-height:1.6'>{body}</div>"))
    report = Report(text, found, det.threshold)
    print(report.summary())
    return report


report = watch("Tell me about Yordanka Donkova.")

## Score text the model already wrote

`score()` checks an answer Qwen3-4B wrote to a prompt; here, the answer above. The probe knows what *its* model knows, so it is meant for Qwen3-4B's own text, not text from other models.

In [ ]:
report = det.score("Tell me about Yordanka Donkova.", report.text)  # the answer written above
for e in sorted(report.entities, key=lambda e: -e.risk)[:15]:
    print(f"{e.risk:4.0%}  {'FLAG' if e.flagged else '    '}  {e.text}")
report  # a Report shows as highlighted text with its summary

## Settings

- Risk is the estimated chance a detail is made up. On held-out text, details given 90–100% risk were false 92–93% of the time.
- `threshold="balanced"` (default): in held-out tests it caught 62–72% of false details and flagged 14–17% of true ones by mistake; 84–87% of flagged details were false.
- `threshold="strict"`: caught 49–62% of false details, flagged 8–11% of true ones; 88–90% of flagged details were false.
- A number between 0 and 1 also works. Details and limits: `docs/LIBRARY.md` in the repository.